# Непрерывный XOR: shared, Gaussian mixture и гауссовская копула

**Самостоятельный ноутбук.** Предыдущий ноутбук и его файлы для запуска не нужны.
Выполните ячейки сверху вниз, затем нажмите **«Запустить / продолжить»**.
По умолчанию — 10 обучений `shared`: два уровня шума и пять seed.

Исследовательский вопрос: способны ли оценки обнаружить зависимость трёх переменных,
когда любые две независимы? Помогает ли получение всех УМО из одной модели плотности?

Это продолжение нашего компактного прототипа по мотивам **SΩI**, а не воспроизведение
официальной реализации статьи. Сеть — тот же тип небольшой MLP с масками и Gaussian skip
connection. Архитектура `shared`, одинаковые веса трёх задач, фиксированные 8000 шагов.
Здесь **3 переменные**, а не 6, как в предыдущем эксперименте. Это упрощает независимую
проверку эталона. Сравнивать абсолютные ошибки между этими двумя задачами как показатель
улучшения алгоритма нельзя.

Сравниваются `shared_direct`, `shared_risk`, `gmm_direct`, `gmm_risk`, `gmm_density`,
`gaussian` и `gaussian_copula`. Две дополнительные строки `oracle_*` — контроль формул:
он использует известное истинное распределение и не является обучаемым конкурентом.

**Сохранение:** в Colab по умолчанию используется Google Drive, папка `soi_xor_experiment`.
После прерывания повторный запуск с теми же настройками продолжает checkpoint.
Кнопка скачивания ZIP работает и для частично выполненной серии.


In [ ]:
import importlib.util
import subprocess
import sys

dependencies = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                "matplotlib": "matplotlib", "sklearn": "scikit-learn", "torch": "torch",
                "ipywidgets": "ipywidgets", "tqdm": "tqdm", "threadpoolctl": "threadpoolctl"}
missing = [pip_name for name, pip_name in dependencies.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

try:
    from google.colab import output
    IN_COLAB = True
    output.enable_custom_widget_manager()
except ImportError:
    IN_COLAB = False
print("Colab:", IN_COLAB)


In [ ]:
from pathlib import Path
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/soi_xor_experiment")
elif IN_COLAB:
    OUTPUT_ROOT = Path("/content/soi_xor_experiment")
else:
    OUTPUT_ROOT = Path.cwd()/"soi_xor_experiment"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Результаты:", OUTPUT_ROOT)
if IN_COLAB and not SAVE_TO_DRIVE:
    print("Без Drive скачивайте ZIP: файлы Colab исчезают после завершения сеанса.")


## Распределение и эталон

Берём независимые равновероятные знаки $A,B\in\{-1,1\}$ и независимые
$\varepsilon_1,\varepsilon_2,\varepsilon_3\sim N(0,1)$:

$$X=\frac{(A+\tau\varepsilon_1,\ B+\tau\varepsilon_2,\ AB+\tau\varepsilon_3)}{\sqrt{1+\tau^2}}.$$

У всех координат нулевое среднее и единичная дисперсия. Любая пара координат независима,
но тройка зависима. Поэтому для этой тройки $\Omega=-TC<0$ и гауссовская оценка по
популяционной ковариации равна нулю. Нормировка масштаба не меняет Ω.

Для $\tau>0$ распределение имеет гладкую плотность — смесь четырёх нормальных компонент.
Эталон вычисляется **из известной плотности**, а не из предсказаний сети:

$$\Omega=\mathbb E_p\left[\log p(X_1)+\log p(X_2)+\log p(X_3)-\log p(X)\right].$$

Используется трёхмерная квадратура Гаусса–Эрмита с проверкой на сетках 32, 48 и 64.
Дополнительно результат проверяется независимым Monte Carlo на 200 000 наблюдениях.
Разность квадратур — диагностика сходимости, **не строгая граница ошибки**.
При несогласии проверок серия останавливается.

В пределе $\tau\to0$ Ω стремится к $-\ln2$. Для ненулевого шума это не точное значение.
Параметры истинной смеси доступны только генератору и внешнему оценщику.
Обучаемая GMM и сеть этих параметров не получают.


In [ ]:
import hashlib
import json
import math
import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logsumexp, ndtr, ndtri, roots_hermitenorm
from scipy.stats import rankdata
from sklearn.mixture import GaussianMixture
import sklearn
from threadpoolctl import threadpool_limits
import torch
from torch import nn
from tqdm.auto import tqdm

VERSION = "xor-soi-1.0"
METHODS = ["gaussian", "gaussian_copula", "gmm_density", "gmm_direct", "gmm_risk",
           "shared_direct", "shared_risk", "oracle_direct", "oracle_risk"]


def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_csv(path, frame):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)


def moments(values):
    values = np.asarray(values, dtype=np.float64)
    return float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))


def gaussian_omega(cov):
    cov = np.asarray(cov, dtype=float)
    sign, ld = np.linalg.slogdet(cov)
    if sign <= 0 or np.linalg.eigvalsh(cov).min() <= 0:
        raise ValueError("Ковариация не положительно определена")
    tc = .5 * (np.log(np.diag(cov)).sum() - ld)
    dtc = .5 * (ld + np.log(np.diag(np.linalg.inv(cov))).sum())
    return float(tc - dtc)


def gaussian_copula_omega(x):
    # Average ranks handle accidental ties; the benchmark is continuous.
    u = (rankdata(x, axis=0, method="average") - .5) / len(x)
    normal = ndtri(u)
    return gaussian_omega(np.cov(normal, rowvar=False, ddof=1))


In [ ]:
class Mixture:
    """Full-covariance Gaussian mixture with analytic noise conditional means."""
    def __init__(self, weights, means, covariances):
        self.weights = np.asarray(weights, dtype=np.float64)
        self.weights = self.weights / self.weights.sum()
        self.means = np.asarray(means, dtype=np.float64)
        self.covs = np.asarray(covariances, dtype=np.float64)
        self.k, self.d = self.means.shape
        self.chol = np.linalg.cholesky(self.covs)
        self.logweights = np.log(self.weights)
        self.cache = {}

    def component_log_prob(self, x, subset=None):
        subset = tuple(range(self.d)) if subset is None else tuple(subset)
        if subset not in self.cache:
            cov = self.covs[:, subset, :][:, :, subset]
            inv = np.linalg.inv(cov)
            const = self.logweights - .5 * (len(subset)*np.log(2*np.pi)
                                             + np.linalg.slogdet(cov)[1])
            self.cache[subset] = self.means[:, subset], inv, const
        mu, inv, const = self.cache[subset]
        diff = np.asarray(x)[:, None, subset] - mu[None]
        return const[None] - .5 * np.einsum("bki,kij,bkj->bk", diff, inv, diff)

    def log_prob(self, x, subset=None):
        return logsumexp(self.component_log_prob(x, subset), axis=1)

    def omega_integrand(self, x):
        # E_q of this expression is Omega(q), for an arbitrary fitted mixture.
        value = -(self.d-2) * self.log_prob(x)
        for i in range(self.d):
            value -= self.log_prob(x, [i])
            value += self.log_prob(x, [j for j in range(self.d) if j != i])
        return value

    def sample(self, n, rng):
        labels = rng.choice(self.k, size=n, p=self.weights)
        return self.means[labels] + np.einsum(
            "bij,bj->bi", self.chol[labels], rng.normal(size=(n, self.d)))

    def prepare_context(self, x):
        # For each Xi, obtain mixture weights/means/variances conditional on X_-i.
        contexts = []
        for i in range(self.d):
            other = [j for j in range(self.d) if j != i]
            cov_other = self.covs[:, other, :][:, :, other]
            cross = self.covs[:, other, i]
            beta = np.linalg.solve(cov_other, cross[..., None])[..., 0]
            diff = x[:, None, other] - self.means[None, :, other]
            mean = self.means[None, :, i] + np.einsum("bki,ki->bk", diff, beta)
            var = self.covs[:, i, i] - np.einsum("ki,ki->k", cross, beta)
            if np.any(var <= 0):
                raise ValueError("Неположительная условная дисперсия смеси")
            contexts.append((self.component_log_prob(x, other), mean, var))
        return contexts

    def predict_noise(self, x, z, sigma, context=None):
        y = x + sigma*z
        noisy_cov = self.covs + sigma*sigma*np.eye(self.d)[None]
        inv = np.linalg.inv(noisy_cov)
        diff = y[:, None, :] - self.means[None]
        logw = self.logweights[None] - .5 * (
            self.d*np.log(2*np.pi) + np.linalg.slogdet(noisy_cov)[1][None]
            + np.einsum("bki,kij,bkj->bk", diff, inv, diff))
        posterior = np.exp(logw-logsumexp(logw, axis=1, keepdims=True))
        joint = sigma*np.einsum("bk,bkj->bj", posterior,
                                np.einsum("bki,kij->bkj", diff, inv))
        marginal, conditional = np.empty_like(y), np.empty_like(y)
        context = self.prepare_context(x) if context is None else context
        for i in range(self.d):
            for out, base, mean, variance in (
                (marginal, self.logweights[None], self.means[None, :, i], self.covs[:, i, i]),
                (conditional, *context[i]),
            ):
                var = variance + sigma*sigma
                delta = y[:, i, None] - mean
                lp = base - .5*(np.log(2*np.pi*var)[None] + delta**2/var[None])
                post = np.exp(lp-logsumexp(lp, axis=1, keepdims=True))
                out[:, i] = sigma*np.sum(post*delta/var[None], axis=1)
        return joint, marginal, conditional

    def save(self, path):
        atomic_npz(path, weights=self.weights, means=self.means, covariances=self.covs)

    @classmethod
    def load(cls, path):
        with np.load(path) as f:
            return cls(f["weights"], f["means"], f["covariances"])


In [ ]:
def xor_distribution(tau):
    if tau <= 0:
        raise ValueError("tau должен быть положительным: работаем с непрерывным распределением")
    scale = np.sqrt(1 + tau*tau)
    means = np.array([[a, b, a*b] for a in (-1., 1.) for b in (-1., 1.)])/scale
    cov = np.repeat((tau*tau/(1+tau*tau)*np.eye(3))[None], 4, axis=0)
    return Mixture(np.full(4, .25), means, cov)


def xor_reference(tau, root, orders=(32, 48, 64), mc_n=200000):
    # Gauss-Hermite quadrature integrates the KNOWN density, not a learned model.
    key = hashlib.sha256(json.dumps([tau, list(orders), mc_n]).encode()).hexdigest()[:12]
    path = Path(root) / f"reference_tau{tau:g}_{key}.json"
    if path.exists():
        return json.loads(path.read_text())
    truth = xor_distribution(tau)
    records = []
    for order in orders:
        nodes, weights = roots_hermitenorm(order)
        weights = weights / np.sqrt(2*np.pi)
        zz = np.stack(np.meshgrid(nodes, nodes, nodes, indexing="ij"), axis=-1).reshape(-1, 3)
        ww = (weights[:, None, None]*weights[None, :, None]*weights[None, None, :]).ravel()
        # All four component-conditional expectations are equal by sign symmetry.
        x = truth.means[0] + tau/np.sqrt(1+tau*tau)*zz
        total = 0.
        for start in range(0, len(x), 16384):
            total += float(ww[start:start+16384] @ truth.omega_integrand(x[start:start+16384]))
        records.append(dict(order=int(order), omega=total))
    rng = np.random.default_rng(937201)
    values = []
    for start in range(0, mc_n, 20000):
        x = truth.sample(min(20000, mc_n-start), rng)
        values.append(truth.omega_integrand(x))
    mc_mean, mc_se = moments(np.concatenate(values))
    delta = abs(records[-1]["omega"]-records[-2]["omega"])
    if delta > 2e-4:
        raise RuntimeError(f"Эталон не стабилизировался: tau={tau}, разность сеток={delta}")
    if abs(mc_mean-records[-1]["omega"]) > max(6*mc_se, 5e-4):
        raise RuntimeError("Независимый Monte Carlo не согласуется с квадратурой эталона")
    result = dict(tau=float(tau), omega=records[-1]["omega"], quadratures=records,
                  last_quadrature_difference=delta, mc_mean=mc_mean, mc_se=mc_se, mc_n=mc_n)
    atomic_json(path, result)
    return result


def rank_control(root, n=15000):
    rng = np.random.default_rng(271828)
    cov = .2*np.eye(3)+.8*np.ones((3, 3))
    x = rng.multivariate_normal(np.zeros(3), cov, n)
    rows = []
    for name, data in [("Gaussian", x), ("Half-cube", x*np.sqrt(np.abs(x))), ("CDF", ndtr(x))]:
        rows.append(dict(transform=name, gaussian=gaussian_omega(np.cov(data, rowvar=False)),
                         gaussian_copula=gaussian_copula_omega(data), truth=gaussian_omega(cov)))
    frame = pd.DataFrame(rows)
    atomic_csv(Path(root)/"rank_control.csv", frame)
    return frame


def trapezoid_weights(grid):
    w = np.empty_like(grid)
    w[0], w[-1] = (grid[1]-grid[0])/2, (grid[-1]-grid[-2])/2
    w[1:-1] = (grid[2:]-grid[:-2])/2
    return w


def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


In [ ]:
class Denoiser(nn.Module):
    def __init__(self, n, width, split, sigma_min, sigma_max):
        super().__init__()
        self.n, self.split = n, split
        self.umin, self.umax = math.log(sigma_min), math.log(sigma_max)
        self.networks = nn.ModuleList([
            nn.Sequential(nn.Linear(3 * n + 9, width), nn.SiLU(),
                          nn.Linear(width, width), nn.SiLU(), nn.Linear(width, n))
            for _ in range(3 if split else 1)
        ])
        for net in self.networks:
            nn.init.zeros_(net[-1].weight)
            nn.init.zeros_(net[-1].bias)

    def forward(self, x, noise, sigma):
        # The three families receive exactly the permitted observations.
        b, n = x.shape
        sigma = sigma.reshape(b, 1)
        y = x + sigma * noise
        yn = y / torch.sqrt(1 + sigma.square())
        eye = torch.eye(n, dtype=x.dtype, device=x.device)[:, None, :]
        joint = yn[None]
        marginal = yn[None] * eye
        conditional = x[None] * (1 - eye) + yn[None] * eye
        values = torch.cat([joint, marginal, conditional], dim=0)
        noisy = torch.cat([torch.ones_like(joint), eye.expand(-1, b, -1),
                           eye.expand(-1, b, -1)], dim=0)
        observed = torch.cat([torch.ones_like(joint), eye.expand(-1, b, -1),
                              torch.ones_like(conditional)], dim=0)
        u = 2 * (torch.log(sigma) - self.umin) / (self.umax - self.umin) - 1
        frequencies = torch.tensor([1., 2., 4., 8.], device=x.device)
        phase = math.pi * u * frequencies
        time_features = torch.cat([u, phase.sin(), phase.cos()], dim=-1)
        f = torch.cat([values, noisy, observed,
                       time_features[None].expand(2*n + 1, -1, -1)], dim=-1)
        if self.split:
            groups = (f[:1], f[1:1+n], f[1+n:])
            residual = torch.cat([
                net(a.reshape(-1, a.shape[-1])).reshape(a.shape[0], b, n)
                for net, a in zip(self.networks, groups)], dim=0)
        else:
            residual = self.networks[0](f.reshape(-1, f.shape[-1]))
            residual = residual.reshape(2*n + 1, b, n)
        # Identical Gaussian marginal skip connection in EVERY architecture.
        # Gaussian initialization is retained from our prototype; this is not upstream SΩI.
        base = (sigma / (1 + sigma.square()) * y)[None]
        scale = (sigma / (1 + sigma.square()))[None]
        pred = base + scale * residual
        ix = torch.arange(n, device=x.device)
        j = pred[0]
        m = pred[1+ix, :, ix].T
        c = pred[1+n+ix, :, ix].T
        return j, m, c


@torch.no_grad()
def predict_network(model, x, z, sigma, batch_size):
    model.eval()
    device = next(model.parameters()).device
    outputs = [[], [], []]
    for start in range(0, len(x), batch_size):
        xt = torch.as_tensor(x[start:start+batch_size], dtype=torch.float32, device=device)
        zt = torch.as_tensor(z[start:start+batch_size], dtype=torch.float32, device=device)
        st = torch.full((len(xt), 1), sigma, dtype=torch.float32, device=device)
        for buf, value in zip(outputs, model(xt, zt, st)):
            buf.append(value.cpu().numpy().astype(np.float64))
    return tuple(np.concatenate(buf) for buf in outputs)


def evaluate_predictors(predictors, x, oracle, cfg, seed, grid_size=None, progress=False):
    grid = np.linspace(np.log(cfg["sigma_min"]), np.log(cfg["sigma_max"]),
                       grid_size or cfg["grid_size"])
    weights = trapezoid_weights(grid)
    rng = np.random.default_rng(seed)
    contexts = {name: p.prepare_context(x) for name, p in predictors.items() if isinstance(p, Mixture)}
    oracle_context = oracle.prepare_context(x)
    sums = {name: {} for name in predictors}
    curves = []
    sq = lambda a: np.square(a).sum(axis=1)
    iterator = tqdm(list(zip(grid, weights)), desc="Оценка на независимом тесте", disable=not progress)
    for u, weight in iterator:
        sigma = float(np.exp(u))
        z = rng.normal(size=x.shape)
        oj, om, oc = oracle.predict_noise(x, z, sigma, oracle_context)
        for name, predictor in predictors.items():
            if predictor is oracle:
                j, m, c = oj, om, oc
            elif isinstance(predictor, Mixture):
                j, m, c = predictor.predict_noise(x, z, sigma, contexts[name])
            else:
                j, m, c = predict_network(predictor, x, z, sigma, cfg["eval_batch"])
            tc, dtc = sq(j-m), sq(j-c)
            rm, rj, rc = sq(z-m), sq(z-j), sq(z-c)
            ej, em, ec = sq(j-oj), sq(m-om), sq(c-oc)
            oracle_risk = sq(z-om)+sq(z-oc)-2*sq(z-oj)
            q = dict(direct=tc-dtc, risk=rm+rc-2*rj, tc=tc, dtc=dtc,
                     risk_tc=rm-rj, risk_dtc=rj-rc,
                     error_joint=ej, error_marginal=em, error_conditional=ec,
                     predicted_risk_bias=em+ec-2*ej,
                     identity_residual=rm+rc-2*rj-oracle_risk-(em+ec-2*ej))
            for key, value in q.items():
                if key not in sums[name]:
                    sums[name][key] = np.zeros(len(x), dtype=np.float64)
                sums[name][key] += weight*value
            curves.append(dict(predictor=name, sigma=sigma, **{key: float(v.mean()) for key, v in q.items()}))
    summaries = {}
    for name, metrics in sums.items():
        summaries[name] = {}
        for key, values in metrics.items():
            summaries[name][key], summaries[name][key+"_se"] = moments(values)
    return summaries, pd.DataFrame(curves), sums


In [ ]:
def fit_mixture(x_train, x_val, cfg, seed, folder):
    path = folder/"gmm.npz"
    if path.exists() and (folder/"gmm_selection.json").exists():
        return Mixture.load(path), json.loads((folder/"gmm_selection.json").read_text())
    candidates, best_model, best_score = [], None, -np.inf
    clock = time.perf_counter()
    for k in cfg["gmm_components"]:
        print(f"GMM: K={k}, выбор по log-likelihood на validation")
        started = time.perf_counter()
        with threadpool_limits(limits=cfg["cpu_threads"]), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fitted = GaussianMixture(n_components=k, covariance_type="full", n_init=cfg["gmm_n_init"],
                                     max_iter=cfg["gmm_max_iter"], reg_covar=1e-5, tol=1e-4,
                                     random_state=70000+seed+k).fit(x_train)
            val_score = float(fitted.score(x_val))
        row = dict(k=int(k), val_log_likelihood=val_score, converged=bool(fitted.converged_),
                   iterations=int(fitted.n_iter_), fit_seconds=time.perf_counter()-started,
                   warnings="; ".join(str(w.message) for w in caught))
        candidates.append(row)
        atomic_csv(folder/"gmm_candidates.csv", pd.DataFrame(candidates))
        if fitted.converged_ and np.isfinite(val_score) and val_score > best_score:
            best_model, best_score = fitted, val_score
    if best_model is None:
        raise RuntimeError("Ни один GMM-кандидат не сошёлся. См. gmm_candidates.csv")
    q = Mixture(best_model.weights_, best_model.means_, best_model.covariances_)
    selection = dict(selected_k=int(best_model.n_components), val_log_likelihood=best_score,
                     fit_seconds=time.perf_counter()-clock, candidates=candidates)
    q.save(path)
    atomic_json(folder/"gmm_selection.json", selection)
    return q, selection


def training_plot(history, truth, path):
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    axes[0].plot(h.step, h.loss)
    axes[0].set(xlabel="Step", ylabel="Noise MSE", title="Shared: training")
    if "val_direct" in h:
        v = h.dropna(subset=["val_direct"])
        axes[1].plot(v.step, v.val_direct, ".-", label="direct")
        axes[1].plot(v.step, v.val_risk, ".-", label="risk")
    axes[1].axhline(truth, color="black", linestyle="--", label="Reference Omega")
    axes[1].set(xlabel="Step", ylabel="Omega, nats", title="Validation (no early stopping)")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    return fig


def fit_network(x_train, x_val, oracle, truth, cfg, seed, folder, device, live):
    torch.manual_seed(1000+seed)
    model = Denoiser(3, cfg["width"], False, cfg["sigma_min"], cfg["sigma_max"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    x_train = x_train.astype(np.float32)
    rng = np.random.default_rng(40000+seed)
    checkpoint = folder/"checkpoint.pt"
    start, history, elapsed, handle = 0, [], 0., None
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location=device, weights_only=False)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        rng.bit_generator.state = saved["rng_state"]
        start, history, elapsed = saved["step"], saved["history"], saved["train_seconds"]
        print("Shared: продолжение с шага", start)
    progress = tqdm(range(start, cfg["steps"]), total=cfg["steps"], initial=start, desc=f"Shared, seed={seed}")
    loss_sum, count = 0., 0
    sync_device(device)
    clock = time.perf_counter()
    for step0 in progress:
        model.train()
        ids = rng.integers(len(x_train), size=cfg["batch_size"])
        z = rng.normal(size=(cfg["batch_size"], 3)).astype(np.float32)
        u = rng.uniform(np.log(cfg["sigma_min"]), np.log(cfg["sigma_max"]), (cfg["batch_size"], 1))
        xt = torch.as_tensor(x_train[ids], device=device)
        zt = torch.as_tensor(z, device=device)
        st = torch.as_tensor(np.exp(u).astype(np.float32), device=device)
        j, m, c = model(xt, zt, st)
        loss = ((j-zt).square().mean()+(m-zt).square().mean()+(c-zt).square().mean())/3
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Нечисловой loss: {folder}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
        optimizer.step()
        loss_sum += float(loss.detach().cpu())
        count += 1
        step = step0+1
        if step % cfg["log_every"] == 0 or step == cfg["steps"]:
            sync_device(device)
            elapsed += time.perf_counter()-clock
            row = dict(step=step, loss=loss_sum/count, train_seconds=elapsed)
            if step % cfg["validate_every"] == 0 or step == cfg["steps"]:
                val, _, _ = evaluate_predictors({"shared": model}, x_val[:cfg["nn_val_limit"]],
                                                oracle, cfg, 50000+seed, grid_size=16)
                for name in ["direct", "risk", "error_joint", "error_marginal", "error_conditional"]:
                    row["val_"+name] = val["shared"][name]
            history.append(row)
            loss_sum, count = 0., 0
            tmp = checkpoint.with_suffix(".tmp")
            torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                            step=step, history=history, train_seconds=elapsed,
                            rng_state=rng.bit_generator.state), tmp)
            os.replace(tmp, checkpoint)
            atomic_csv(folder/"training.csv", pd.DataFrame(history))
            progress.set_postfix(loss=round(row["loss"], 4))
            if "val_direct" in row:
                fig = training_plot(history, truth, folder/"training.png")
                if live:
                    from IPython.display import display
                    if handle is None:
                        handle = display(fig, display_id=True)
                    else:
                        handle.update(fig)
                plt.close(fig)
            sync_device(device)
            clock = time.perf_counter()
    return model, elapsed


In [ ]:
def run_one(cfg, tau, seed, reference, root, device, live=True):
    job = dict(version=VERSION, tau=float(tau), seed=int(seed), config=cfg,
               device=str(device), torch=str(torch.__version__), sklearn=sklearn.__version__)
    token = hashlib.sha256(json.dumps(job, sort_keys=True).encode()).hexdigest()[:12]
    folder = Path(root)/f"xor_tau{tau:g}_s{seed}_{token}"
    folder.mkdir(parents=True, exist_ok=True)
    if (folder/"result.json").exists():
        print("Готово ранее:", folder.name)
        return json.loads((folder/"result.json").read_text())
    atomic_json(folder/"config.json", job)
    oracle = xor_distribution(tau)
    x_train = oracle.sample(cfg["n_train"], np.random.default_rng(10000+seed))
    x_val = oracle.sample(cfg["n_val"], np.random.default_rng(20000+seed))
    x_test = oracle.sample(cfg["n_test"], np.random.default_rng(30000+seed))
    q, selection = fit_mixture(x_train, x_val, cfg, seed, folder)
    model, train_seconds = fit_network(x_train, x_val, oracle, reference["omega"],
                                       cfg, seed, folder, device, live)
    clock = time.perf_counter()
    summary, curves, observations = evaluate_predictors(
        {"shared": model, "gmm": q, "oracle": oracle}, x_test, oracle, cfg, 60000+seed, progress=live)
    eval_seconds = time.perf_counter()-clock
    atomic_csv(folder/"integrands.csv", curves)
    atomic_npz(folder/"observations.npz", **{f"{name}__{key}": v
               for name, values in observations.items() for key, v in values.items()})
    # A classical plug-in Omega(q): the expectation is taken under q, NOT P.
    rng = np.random.default_rng(80000+seed)
    plugin_values = []
    for start in range(0, cfg["density_mc_n"], 16384):
        sample = q.sample(min(16384, cfg["density_mc_n"]-start), rng)
        plugin_values.append(q.omega_integrand(sample))
    plugin, plugin_se = moments(np.concatenate(plugin_values))
    common = dict(tau=float(tau), seed=int(seed), n_train=cfg["n_train"],
                  n_val=cfg["n_val"], n_test=cfg["n_test"], truth_omega=reference["omega"],
                  reference_grid_difference=reference["last_quadrature_difference"],
                  steps=cfg["steps"], folder=str(folder), gmm_k=selection["selected_k"],
                  shared_parameters=sum(p.numel() for p in model.parameters()),
                  shared_train_seconds=train_seconds, gmm_fit_seconds=selection["fit_seconds"],
                  all_predictors_eval_seconds=eval_seconds)
    results = []
    def add(method, value, se=None, **extra):
        error = float(value-reference["omega"])
        results.append(dict(**common, method=method, omega=float(value), conditional_eval_se=se,
                            signed_error=error, abs_error=abs(error), squared_error=error*error, **extra))
    add("gaussian", gaussian_omega(np.cov(x_train, rowvar=False, ddof=1)))
    add("gaussian_copula", gaussian_copula_omega(x_train))
    add("gmm_density", plugin, plugin_se)
    for name in ["shared", "gmm", "oracle"]:
        s = summary[name]
        for estimator in ["direct", "risk"]:
            add(name+"_"+estimator, s[estimator], s[estimator+"_se"],
                error_joint=s["error_joint"], error_marginal=s["error_marginal"],
                error_conditional=s["error_conditional"], predicted_risk_bias=s["predicted_risk_bias"],
                identity_residual=s["identity_residual"], identity_residual_se=s["identity_residual_se"])
    atomic_json(folder/"evaluation_diagnostics.json", summary)
    atomic_json(folder/"result.json", results)
    return results


def save_summary(results, root):
    frame = pd.DataFrame(results)
    summary = frame.groupby(["tau", "method"], as_index=False).agg(
        seeds=("seed", "count"), omega_mean=("omega", "mean"), omega_sd=("omega", "std"),
        mae=("abs_error", "mean"), mse=("squared_error", "mean"),
        bias=("signed_error", "mean"), truth=("truth_omega", "first"))
    summary["rmse"] = np.sqrt(summary.pop("mse"))
    atomic_csv(Path(root)/"results.csv", frame)
    atomic_csv(Path(root)/"summary.csv", summary)
    return frame, summary


def comparison_plot(results, root):
    df = pd.DataFrame(results)
    taus = sorted(df.tau.unique())
    fig, axes = plt.subplots(len(taus), 2, figsize=(13, 4.8*len(taus)), squeeze=False)
    rng = np.random.default_rng(0)
    for (ax, err_ax), tau in zip(axes, taus):
        part = df[df.tau == tau]
        for pos, method in enumerate(METHODS):
            values = part[part.method == method]
            if values.empty:
                continue
            y = pos+rng.uniform(-.13, .13, len(values))
            color = "#777777" if method.startswith("oracle") else ("#c4572e" if method.startswith("shared") else "#2967a4")
            ax.scatter(values.omega, y, s=22, color=color)
            err_ax.scatter(values.abs_error, y, s=22, color=color)
        ax.axvline(float(part.truth_omega.iloc[0]), color="black", linestyle="--", label="Reference")
        for panel in (ax, err_ax):
            panel.set_yticks(range(len(METHODS)), METHODS, fontsize=9)
            panel.invert_yaxis()
            panel.grid(axis="x", alpha=.2)
        ax.set(xlabel="Omega, nats", title=f"XOR, tau={tau:g}: each dot is one seed")
        ax.legend(fontsize=8)
        err_ax.set(xlabel="Absolute error, nats", title="Oracle uses the true distribution")
    fig.tight_layout()
    fig.savefig(Path(root)/"comparison.png", dpi=150)
    return fig


def run_suite(cfg, taus, seeds, root, device, live=True):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    protocol = dict(version=VERSION, config=cfg, taus=list(taus), seeds=list(seeds),
                    device=str(device), python=platform.python_version(), torch=str(torch.__version__),
                    numpy=np.__version__, sklearn=sklearn.__version__)
    token = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:12]
    suite_root = root/token
    suite_root.mkdir(parents=True, exist_ok=True)
    atomic_json(suite_root/"protocol.json", protocol)
    references = {tau: xor_reference(tau, suite_root) for tau in taus}
    atomic_csv(suite_root/"references.csv", pd.DataFrame([
        {k: v for k, v in ref.items() if k != "quadratures"} for ref in references.values()]))
    control = rank_control(suite_root)
    if live:
        from IPython.display import display
        display(control.round(6))
    results = []
    for tau in taus:
        for seed in seeds:
            print(f"\nXOR tau={tau:g}, seed={seed}, reference={references[tau]['omega']:.6f}")
            try:
                results.extend(run_one(cfg, tau, seed, references[tau], suite_root, device, live))
            except Exception as error:
                with (suite_root/"failures.jsonl").open("a", encoding="utf-8") as f:
                    f.write(json.dumps(dict(tau=tau, seed=seed, error=repr(error)), ensure_ascii=False)+"\n")
                raise
            frame, summary = save_summary(results, suite_root)
            fig = comparison_plot(results, suite_root)
            if live:
                display(fig)
                display(summary[["tau", "method", "seeds", "mae", "bias"]].round(5))
            plt.close(fig)
    print("Готово. Результаты:", suite_root)
    return frame, summary, suite_root


## Что именно оценивают методы

Для $Y=X+\sigma Z$, $Z\sim N(0,I)$ предсказывается шум:
$J_i=\mathbb E[Z_i\mid Y]$, $M_i=\mathbb E[Z_i\mid Y_i]$,
$C_i=\mathbb E[Z_i\mid Y_i,X_{-i}]$.
Интегралы считаются по $u=\log\sigma$:

$$\widehat\Omega_{direct}=\int\mathbb E\big[\|\widehat J-\widehat M\|^2-
\|\widehat J-\widehat C\|^2\big]\,du,$$

$$\widehat\Omega_{risk}=\int\mathbb E\big[\|Z-\widehat M\|^2+
\|Z-\widehat C\|^2-2\|Z-\widehat J\|^2\big]\,du.$$

| Метод | Как получается оценка |
|---|---|
| `shared_direct`, `shared_risk` | УМО приближены общей MLP; обе формулы оцениваются на одном тесте и одних шумах. |
| `gmm_direct`, `gmm_risk` | УМО аналитически вычислены из **обученной** GMM; тест и шумы совпадают с shared. |
| `gmm_density` | Оценивается Ω самой обученной плотности $q$: выборка генерируется из $q$, логарифмы плотностей считаются по $q$. Диффузионное интегрирование не используется. |
| `gaussian` | Гауссовская формула по выборочной ковариации train. |
| `gaussian_copula` | Нормализация train через ранги и нормальные квантили, затем гауссовская формула. |
| `oracle_direct`, `oracle_risk` | Те же интегралы, но УМО получены из **истинной** смеси; контроль реализации и численного оценивания. |

GMM: полные ковариации, кандидаты $K\in\{1,2,4,8\}$. Все обучаются на одной train-выборке.
Число компонент выбирается по среднему log-likelihood на отдельной validation-выборке.
Истинная Ω, test и ошибки УМО не используются для выбора K. Все кандидаты, включая
предупреждения и несходимость, записываются в `gmm_candidates.csv`.
На этом распределении GMM имеет преимущество правильно выбранного семейства:
её успех не доказывает универсального превосходства над нейросетью.

Для shared используется последний checkpoint фиксированного бюджета; validation-график
показывает динамику и не выбирает лучшую точку. Gaussian skip connection сохранён из
предыдущего прототипа. В начале все три предсказания совпадают и обе оценки Ω равны нулю,
хотя у XOR истинная Ω отрицательна.

`conditional_eval_se` учитывает случайность оценивания **при фиксированной обученной модели**.
Он не учитывает разброс обучения и смещение метода. Для `gaussian` и `gaussian_copula`
этот столбец пустой. О качестве метода судим по пяти независимым seed, MAE, RMSE и смещению.
На итоговом рисунке каждая точка — отдельный seed, без условных доверительных интервалов.

Ошибки относительно истинных УМО сохраняются отдельно. Для `risk` диагностируем
$B=E_M+E_C-2E_J$. Это компонент смещения из-за модели; к нему добавляются численная
ошибка интегрирования и случайность теста. Контроль `oracle_*` помогает их заметить.


In [ ]:
# Цвет третьей координаты показывает зависимость, которую не видно по одной паре.
from IPython.display import display
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, tau in zip(axes, [.5, 1.]):
    x = xor_distribution(tau).sample(2000, np.random.default_rng(1729))
    dots = ax.scatter(x[:, 0], x[:, 1], c=x[:, 2], cmap="coolwarm", s=7,
                      alpha=.65, vmin=-2, vmax=2)
    ax.set(xlabel="X1", ylabel="X2", title=f"Continuous XOR, tau={tau}")
    fig.colorbar(dots, ax=ax, label="X3")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT/"xor_preview.png", dpi=140)
display(fig)
plt.close(fig)


## Запуск

**Основной режим:** $\tau=0.5,1.0$, seed 10–14, 15 000 train, 2048 validation,
8192 test, 8000 шагов, batch 64, ширина MLP 64. В обоих случаях выполняются все методы.
GMM и эталон работают на CPU; CUDA/MPS используется для нейросети. Для `gmm_density`
генерируется 100 000 дополнительных объектов из обученной смеси — это вычисление оценки,
а не дополнительные обучающие данные.

**Проверка:** два коротких запуска по 200 шагов. Нужна только для проверки среды и файлов.
Не используйте её для научных выводов. После проверки выберите основной режим.

Время обучения, параметры среды и конфигурация записываются. После остановки повторите
запуск с теми же настройками и тем же устройством. Сеть продолжит последний сохранённый
checkpoint; завершённые задания пропускаются. Смена численных настроек создаёт другую серию.
Во время GMM checkpoint ещё нет, но завершённая выбранная смесь сохраняется до обучения сети.

Пять seed меняют и обучающую выборку, и инициализацию. Два значения tau используют
согласованные случайные числа; их нельзя считать десятью независимыми повторениями одного эффекта.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, FileLink
import shutil

device_options = ["cpu"]
if torch.cuda.is_available():
    device_options.insert(0, "cuda")
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_options.insert(0, "mps")

profile_ui = widgets.Dropdown(options=[("Основной: 10 запусков", "main"),
                                     ("Проверка: 2 коротких запуска", "smoke")], value="main", description="Режим")
device_ui = widgets.Dropdown(options=device_options, value=device_options[0], description="Устройство")
run_button = widgets.Button(description="Запустить / продолжить", button_style="primary",
                            layout=widgets.Layout(width="230px"))
zip_button = widgets.Button(description="Скачать ZIP")
ui_output = widgets.Output()


def selected_config():
    cfg = dict(width=64, sigma_min=.005, sigma_max=100., grid_size=64, eval_batch=256,
               steps=8000, lr=.002, batch_size=64, log_every=250, validate_every=500,
               n_train=15000, n_val=2048, nn_val_limit=256, n_test=8192,
               gmm_components=[1, 2, 4, 8], gmm_n_init=3, gmm_max_iter=300,
               density_mc_n=100000, cpu_threads=min(4, os.cpu_count() or 1))
    seeds = list(range(10, 15))
    if profile_ui.value == "smoke":
        cfg.update(steps=200, log_every=100, validate_every=200, n_train=1000,
                   n_val=512, n_test=512, nn_val_limit=128, grid_size=32,
                   gmm_n_init=2, density_mc_n=10000)
        seeds = [10]
    return cfg, [.5, 1.], seeds


def start_experiment(_):
    global RESULTS, SUMMARY, RUN_ROOT, LAST_CONFIG
    run_button.disabled = zip_button.disabled = True
    with ui_output:
        ui_output.clear_output(wait=True)
        try:
            cfg, taus, seeds = selected_config()
            LAST_CONFIG = cfg
            device = torch.device(device_ui.value)
            if device.type == "cpu":
                torch.set_num_threads(cfg["cpu_threads"])
            RESULTS, SUMMARY, RUN_ROOT = run_suite(cfg, taus, seeds, OUTPUT_ROOT, device, live=True)
            print("Скачайте ZIP или пришлите summary.csv, results.csv, references.csv и comparison.png.")
        finally:
            run_button.disabled = zip_button.disabled = False


def download_results(_):
    with ui_output:
        archive = shutil.make_archive(str(OUTPUT_ROOT.parent/"soi_xor_results"), "zip", OUTPUT_ROOT)
        if IN_COLAB:
            from google.colab import files
            files.download(archive)
        else:
            display(FileLink(archive))


run_button.on_click(start_experiment)
zip_button.on_click(download_results)
display(widgets.VBox([profile_ui, device_ui, widgets.HBox([run_button, zip_button]), ui_output]))


## Какие файлы смотреть

| Файл | Содержание |
|---|---|
| `summary.csv` | MAE, RMSE, смещение и разброс по seed для каждого tau и метода. |
| `results.csv` | Каждая оценка отдельно, её условная SE, время обучения, число компонент GMM и ошибки УМО. |
| `comparison.png` | Все seed, эталон и абсолютные ошибки. |
| `references.csv`, `reference_*.json` | Эталонная Ω и две независимые численные проверки. |
| `rank_control.csv` | Быстрый контроль: Gaussian, Half-cube и CDF дают одинаковую ранговую оценку. Это один контрольный пример, не полноценный benchmark. |
| `gmm_candidates.csv` в папке запуска | Выбор K по валидационному правдоподобию и сведения о сходимости. |
| `training.csv`, `training.png`, `checkpoint.pt` в папке запуска | Динамика shared и продолжение обучения. |
| `integrands.csv`, `observations.npz` | Интегранды и вклады тестовых объектов для последующего анализа. |

Сначала проверьте `oracle_*`: если он плохо согласуется с эталоном, нужно разобраться
с численным оцениванием. Затем сравните `shared_*` и `gmm_*` по seed, учитывая время.
Различие `gmm_density` и `gmm_direct/risk` допустимо: первое оценивает Ω модели $q$,
последние подставляют её денойзеры в ожидания по независимым данным из $p$.
При $q=p$ и точном интегрировании они совпадают.

Не выбирайте метод, checkpoint или K по финальному тесту. Если потребуется настройка
после просмотра этой серии, подтвердите выводы на новых seed.

Источники:
- [SΩI, статья и приложение E](https://arxiv.org/html/2402.05667v3).
- [Gaussian-copula estimation, Ince et al.](https://github.com/robince/gcmi).
- [GaussianMixture, scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html).
- [MMG: разность MMSE и ортогональная форма](https://arxiv.org/html/2509.20609v2).
